In [1]:
###############------------#######################
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import typing
import warnings
import typing_extensions as tx
from tensorflow.keras import layers
from tensorflow.keras import models
import numpy as np
import tensorflow_addons as tfa
import glob, warnings
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from tensorflow.keras import callbacks
from keras.callbacks import Callback
import imageio
from tensorflow import keras
from joblib import Parallel, delayed
import efficientnet.tfkeras
from tensorflow.keras.models import load_model

/home/kannika/miniconda3/envs/AI/lib/python3.6/site-packages/tensorflow_addons/utils/ensure_tf_install.py:68: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.2.0 and strictly below 2.3.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.6.2 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/addons
  UserWarning,


In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]= "0"

## Check Dataset

In [3]:
data_path = "/media/tohn/HDD/VISION_dataset"

In [4]:
dataframe = pd.read_csv(f'{data_path}/Traindf_fold4_8_v1.csv')
#validation
valframe = pd.read_csv(f'{data_path}/Valdf_fold3_v1.csv')
print(valframe.shape)
print(dataframe.shape)
dataframe

(656, 27)
(4601, 27)


,Unnamed: 0,Case,Abs Position,Sub Position,Class,Path Full,Path Crop,Views,fold,15AB,...,Rwidth,Rheight,Sub_class_New,tag_AjNit,tag_AjWan,Test150,Spilt,Sub_Position_New,filename,Sub_Position_Label
0,34,42,P5,P52,Abnormal,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,FP-C,7,MildFattyLiver,...,0.633663,0.651394,AB01,NaN,NaN,False,Train,None,AB01 P5-2 C042.JPG,P7
1,35,42,P1,P1,Abnormal,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,FP-A,7,ModerateFattyLiver,...,0.541516,0.422222,AB02,NaN,NaN,False,Train,P1,AB02 P1 C042.JPG,P1
2,36,42,P2,P2,Abnormal,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,FP-A,7,ModerateFattyLiver,...,0.494585,0.610101,AB02,NaN,NaN,False,Train,P2,AB02 P2 C042.JPG,P2
3,37,42,P5,P51,Abnormal,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,FP-C,7,ModerateFattyLiver,...,0.494545,0.668687,AB02,NaN,NaN,False,Train,P6,AB02 P5-1 C042.JPG,P6
4,38,42,P4,P42,Abnormal,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,/media/tohn/HDD/VISION_dataset/USAI/ABnormal01...,FP-B,7,ModerateFattyLiver,...,0.477148,0.690909,AB02,NaN,NaN,False,Train,P5,AB02 P4-2 C042.JPG,P5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4596,6564,344,P3,P32,Normal,/media/tohn/HDD/VISION_dataset/USAI/US images ...,/media/tohn/HDD/VISION_dataset/USAI/US images ...,FP-B,4,Normal,...,NaN,NaN,Normal,NaN,NaN,False,Train,P9,P3-2.Case 344.JPG,P9
4597,6565,344,P4,P42,Normal,/media/tohn/HDD/VISION_dataset/USAI/US images ...,/media/tohn/HDD/VISION_dataset/USAI/US images ...,FP-B,4,Normal,...,NaN,NaN,Normal,NaN,NaN,False,Train,P5,P4-2.Case 344.JPG,P5
4598,6566,344,P5,P52,Normal,/media/tohn/HDD/VISION_dataset/USAI/US images ...,/media/tohn/HDD/VISION_dataset/USAI/US images ...,FP-C,4,Normal,...,NaN,NaN,Normal,NaN,NaN,False,Train,None,P5-2.Case 344.JPG,P7
4599,6567,344,P6,P61,Normal,/media/tohn/HDD/VISION_dataset/USAI/US images ...,/media/tohn/HDD/VISION_dataset/USAI/US images ...,FP-C,4,Normal,...,NaN,NaN,Normal,NaN,NaN,False,Train,P14,P6.Case 344.JPG,P14


In [12]:
print(len(set(dataframe['Sub_class_New'])))
print(set(dataframe['Sub_class_New']))

15
{'AB04', 'AB03', 'AB05', 'AB11', 'AB01', 'AB06', 'AB02', 'AB083', 'AB12', 'AB09', 'AB10', 'AB082', 'AB07', 'AB081', 'Normal'}


In [14]:
print(len(set(valframe['Sub_class_New'])))
print(set(valframe['Sub_class_New']))

15
{'AB04', 'AB03', 'AB05', 'AB11', 'AB01', 'AB12', 'AB02', 'AB06', 'AB083', 'AB09', 'AB10', 'AB082', 'AB07', 'AB081', 'Normal'}


In [17]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [18]:
BATCH_SIZE = 16
height=width = 456

train_datagen = ImageDataGenerator(
      rescale=1./255,
      rotation_range=30,
      width_shift_range=0.2,
      height_shift_range=0.2,
      brightness_range=[0.5, 1.5],
      shear_range=0.4,
      zoom_range=0.2,
      horizontal_flip=False,
      fill_mode='nearest')

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe = dataframe,
    directory = None,
    x_col = 'Path Crop',
    y_col = 'Sub_class_New',
    target_size = (height, width),
    batch_size=BATCH_SIZE,
    color_mode= 'rgb',
    class_mode='categorical')

val_generator = valid_datagen.flow_from_dataframe(
    dataframe = valframe,
    directory = None,
    x_col = 'Path Crop',
    y_col = 'Sub_class_New',
    target_size = (height, width),
    batch_size=BATCH_SIZE,
    color_mode= 'rgb',
    class_mode='categorical')

Found 4601 validated image filenames belonging to 15 classes.
Found 656 validated image filenames belonging to 15 classes.


### TEST Code :Resume Model

In [3]:
import tensorflow as tf
from keras import layers
from tensorflow.keras import optimizers
from tensorflow.keras import callbacks
from efficientnet.keras import center_crop_and_resize, preprocess_input
import tensorflow.keras as keras
from keras import models
from tensorflow.keras.models import Model, model_from_json
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense#, Conv2D, BatchNormalization
from tensorflow.keras.applications import EfficientNetB5 as Net

In [4]:
path_model = "/media/tohn/HDD2/Model_unlearn/Models_USAI/ML-unlearn/R1_balanced/unfreezeB4-B7/on_epoch_end/modelEffNetB5_ML-unlearn_unfreezeB4-B7-R1_balanced_last.h5"
model = load_model(path_model)
height = width = model.input_shape[1]
input_shape = (height, width, 3)

In [5]:
model.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 30,735
Non-trainable params: 28,513,520
_________________________________________________________________


In [6]:
model.input_shape[1]

456

In [7]:
print(height, width)

456 456


### TEST Code unfrrze FC layer ==> EffNetB5

In [23]:
from tensorflow.keras.applications import EfficientNetB5 as Net

In [8]:
height = width = 456
input_shape = (height, width, 3)
# loading pretrained conv base model
conv_base = Net(weights='imagenet', include_top=False, input_shape=input_shape)
# create new model with a new classification layer
x = conv_base.output  
global_average_layer = layers.GlobalAveragePooling2D(name = 'head_pooling')(x)
dropout_layer_1 = layers.Dropout(0.50,name = 'head_dropout')(global_average_layer)
prediction_layer = layers.Dense(15, activation='softmax',name = 'prediction_layer')(dropout_layer_1)
## Create model 
model = models.Model(inputs= conv_base.input, outputs=prediction_layer) 
### Unfreeze FC layers 
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.trainable_weights))
print('[INFO]: Freezing hidden layers...')
for layer in conv_base.layers:
    layer.trainable = False

print('This is the number of trainable layers '
   'after freezing the conv base:', len(model.trainable_weights))
print('-'*125)

This is the number of trainable layers before freezing the conv base: 506
[INFO]: Freezing hidden layers...
This is the number of trainable layers after freezing the conv base: 2
-----------------------------------------------------------------------------------------------------------------------------


In [9]:
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
rescaling_2 (Rescaling)         (None, 456, 456, 3)  0           input_3[0][0]                    
__________________________________________________________________________________________________
normalization_2 (Normalization) (None, 456, 456, 3)  7           rescaling_2[0][0]                
__________________________________________________________________________________________________
stem_conv_pad (ZeroPadding2D)   (None, 457, 457, 3)  0           normalization_2[0][0]            
____________________________________________________________________________________________

In [28]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: input_3 | Trainable: False
Layer: rescaling_2 | Trainable: False
Layer: normalization_2 | Trainable: False
Layer: stem_conv_pad | Trainable: False
Layer: stem_conv | Trainable: False
Layer: stem_bn | Trainable: False
Layer: stem_activation | Trainable: False
Layer: block1a_dwconv | Trainable: False
Layer: block1a_bn | Trainable: False
Layer: block1a_activation | Trainable: False
Layer: block1a_se_squeeze | Trainable: False
Layer: block1a_se_reshape | Trainable: False
Layer: block1a_se_reduce | Trainable: False
Layer: block1a_se_expand | Trainable: False
Layer: block1a_se_excite | Trainable: False
Layer: block1a_project_conv | Trainable: False
Layer: block1a_project_bn | Trainable: False
Layer: block1b_dwconv | Trainable: False
Layer: block1b_bn | Trainable: False
Layer: block1b_activation | Trainable: False
Layer: block1b_se_squeeze | Trainable: False
Layer: block1b_se_reshape | Trainable: False
Layer: block1b_se_reduce | Trainable: False
Layer: block1b_se_expand | Trainable: Fa

--------

### TEST Code Unfreeze block5a_se_excite

In [3]:
from tensorflow.keras.models import Model, model_from_json

path_modelJson = "/media/tohn/HDD/Model_unlearn/Models_USAI/R1/models/modelEffNetB5_EffNet-base_transfer-R1.json"
path_modelweights = "/media/tohn/HDD/Model_unlearn/Models_USAI/R1/models/modelEffNetB5_EffNet-base_transfer-R1.weights.h5"

##load json and create model --------------------
json_file = open(path_modelJson, 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)
#load weights into new model  -------------------

model.load_weights(path_modelweights)
print("Loaded model from disk")

model.summary()

Loaded model from disk
Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
rescaling (Rescaling)           (None, 456, 456, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
normalization (Normalization)   (None, 456, 456, 3)  7           rescaling[0][0]                  
__________________________________________________________________________________________________
stem_conv_pad (ZeroPadding2D)   (None, 457, 457, 3)  0           normalization[0][0]              
_______________________________________________________________________

In [9]:
height = width = model.input_shape[1]
input_shape = (height, width, 3)
print(input_shape)

(456, 456, 3)


In [5]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: input_1 | Trainable: True
Layer: rescaling | Trainable: False
Layer: normalization | Trainable: False
Layer: stem_conv_pad | Trainable: False
Layer: stem_conv | Trainable: False
Layer: stem_bn | Trainable: False
Layer: stem_activation | Trainable: False
Layer: block1a_dwconv | Trainable: False
Layer: block1a_bn | Trainable: False
Layer: block1a_activation | Trainable: False
Layer: block1a_se_squeeze | Trainable: False
Layer: block1a_se_reshape | Trainable: False
Layer: block1a_se_reduce | Trainable: False
Layer: block1a_se_expand | Trainable: False
Layer: block1a_se_excite | Trainable: False
Layer: block1a_project_conv | Trainable: False
Layer: block1a_project_bn | Trainable: False
Layer: block1b_dwconv | Trainable: False
Layer: block1b_bn | Trainable: False
Layer: block1b_activation | Trainable: False
Layer: block1b_se_squeeze | Trainable: False
Layer: block1b_se_reshape | Trainable: False
Layer: block1b_se_reduce | Trainable: False
Layer: block1b_se_expand | Trainable: False
L

In [6]:
# height = width = model.input_shape[1]
# input_shape = (height, width, 3)
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.trainable_weights))
model.trainable = True
set_trainable = False
for layer in model.layers:
    if layer.name == 'block5a_se_excite':
        set_trainable = True
    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False
print('This is the number of trainable layers '
      'after freezing the block5a_se_excite Layer:', len(model.trainable_weights))

This is the number of trainable layers before freezing the conv base: 2
This is the number of trainable layers after freezing the block5a_se_excite Layer: 242


In [7]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: input_1 | Trainable: False
Layer: rescaling | Trainable: False
Layer: normalization | Trainable: False
Layer: stem_conv_pad | Trainable: False
Layer: stem_conv | Trainable: False
Layer: stem_bn | Trainable: False
Layer: stem_activation | Trainable: False
Layer: block1a_dwconv | Trainable: False
Layer: block1a_bn | Trainable: False
Layer: block1a_activation | Trainable: False
Layer: block1a_se_squeeze | Trainable: False
Layer: block1a_se_reshape | Trainable: False
Layer: block1a_se_reduce | Trainable: False
Layer: block1a_se_expand | Trainable: False
Layer: block1a_se_excite | Trainable: False
Layer: block1a_project_conv | Trainable: False
Layer: block1a_project_bn | Trainable: False
Layer: block1b_dwconv | Trainable: False
Layer: block1b_bn | Trainable: False
Layer: block1b_activation | Trainable: False
Layer: block1b_se_squeeze | Trainable: False
Layer: block1b_se_reshape | Trainable: False
Layer: block1b_se_reduce | Trainable: False
Layer: block1b_se_expand | Trainable: False


In [8]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
rescaling (Rescaling)           (None, 456, 456, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
normalization (Normalization)   (None, 456, 456, 3)  7           rescaling[0][0]                  
__________________________________________________________________________________________________
stem_conv_pad (ZeroPadding2D)   (None, 457, 457, 3)  0           normalization[0][0]              
______________________________________________________________________________________________

Total params: 28,544,262
Trainable params: 26,109,423
Non-trainable params: 2,434,839
__________________________________________________________________________________________________


--------

### TEST CODE: : Load ML Unlearn unfreezeB4 Model to Finetune Stage: Unfreeze FC Layers

In [18]:
path_model = "/media/tohn/SSD/Machine-Unlearning-USAI-CCA-2024/models/modelEffNetB5_Unlearning_unfreezeB4_R2.h5"
old_model = load_model(path_model)
old_model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_2 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
input_3 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
concatenate (Concatenate)       (None, 456, 456, 6)  0           input_2[0][0]                    
                                                                 input_3[0][0]                    
__________________________________________________________________________________________________
conv2d (Conv2D)                 (None, 456, 456, 3)  21          concatenate[0][0]            

In [19]:
for layer in old_model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: input_2 | Trainable: True
Layer: input_3 | Trainable: True
Layer: concatenate | Trainable: True
Layer: conv2d | Trainable: True
Layer: batch_normalization | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: head_pooling | Trainable: True
Layer: head_dropout | Trainable: True
Layer: prediction_layer | Trainable: True


In [21]:
layers = old_model.layers[5].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: True
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Layer

In [22]:
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import GlobalAveragePooling2D
from keras.layers import Dropout
from keras.layers import Dense
from keras.models import Model

In [35]:
# Step 2: ดึง EfficientNet-B5 จากโมเดลเดิม
efficientnet_b5_layer = old_model.get_layer('efficientnet-b5')  # ดึงเฉพาะ EfficientNet-B5
efficientnet_b5_layer.trainable =  False #True  # สามารถเลือกให้ฝึกต่อได้ หรือจะ freeze ก็ได้

# Step 3: สร้าง input layer ใหม่ที่มีขนาด (456, 456, 3)
height=width = 456
new_input = Input(shape=(height, width, 3), name='new_input')
input_shape = (height, width, 3)

# Step 4: ต่อเลเยอร์ EfficientNet-B5 เข้ากับ input ใหม่
x = efficientnet_b5_layer(new_input)

# Step 5: เพิ่มเลเยอร์ Global Average Pooling, Dropout, และเลเยอร์ output (15 classes)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)  # สามารถปรับ dropout ตามความเหมาะสม
output = Dense(15, activation='softmax', name='prediction_layer')(x)  # 15 classes ใช้ softmax สำหรับ multi-class classification

# Step 6: สร้างโมเดลใหม่
new_model = Model(inputs=new_input, outputs=output)

# Step 7: Compile โมเดลใหม่ (ปรับ optimizer และ loss function ตามที่ต้องการ)
new_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

#showing before&after freezing
print('This is the number of trainable layers '
      'before freezing the conv base:', len(new_model.trainable_weights))
#conv_base.trainable = False  # freeze เพื่อรักษา convolutional base's weight
# for layer in new_model.layers:
#     layer.trainable = False
# print('This is the number of trainable layers '
#       'after freezing the conv base:', len(new_model.trainable_weights))

This is the number of trainable layers before freezing the conv base: 2


In [36]:
new_model.summary()

Model: "model_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d_3 ( (None, 2048)              0         
_________________________________________________________________
dropout_3 (Dropout)          (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 30,735
Non-trainable params: 28,513,520
_________________________________________________________________


In [37]:
for layer in new_model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: False
Layer: global_average_pooling2d_3 | Trainable: True
Layer: dropout_3 | Trainable: True
Layer: prediction_layer | Trainable: True


In [26]:
layers = new_model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: False
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Laye

In [16]:
new_model.layers[1]

In [43]:
for layer in new_model.layers:
    layer.trainable = False
print('This is the number of trainable layers '
      'after freezing the conv base:', len(new_model.trainable_weights))

In [29]:
new_model.summary()

Model: "model_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d_2 ( (None, 2048)              0         
_________________________________________________________________
dropout_2 (Dropout)          (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 0
Non-trainable params: 28,544,255
_________________________________________________________________


In [30]:
for layer in new_model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: False
Layer: efficientnet-b5 | Trainable: False
Layer: global_average_pooling2d_2 | Trainable: False
Layer: dropout_2 | Trainable: False
Layer: prediction_layer | Trainable: False


In [31]:
layers = new_model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: False
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Laye

In [41]:
# for layer in new_model.layers:
#     print(layer.name)
#     if layer.name == "efficientnet-b5":
#         layer.trainable = False

In [42]:
# for layer in new_model.layers:
#     print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

> #### ใช้ Code นี้ 

In [38]:
layers = new_model.layers[1].layers
for innerlayer in layers:
    innerlayer.trainable = False
    
fc_layer = new_model.get_layer("prediction_layer")
fc_layer.trainable = True

In [39]:
for layer in new_model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: False
Layer: global_average_pooling2d_3 | Trainable: True
Layer: dropout_3 | Trainable: True
Layer: prediction_layer | Trainable: True


In [40]:
new_model.summary()

Model: "model_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d_3 ( (None, 2048)              0         
_________________________________________________________________
dropout_3 (Dropout)          (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 30,735
Non-trainable params: 28,513,520
_________________________________________________________________


### TEST CODE: Load ML Unlearn unfreezeB4 Model to Finetune Stage: Unfreeze Block4

In [4]:
from tensorflow.keras.models import Model, model_from_json

In [9]:
path_modelJson = "/home/yupaporn/codes/Machine-Unlearning-USAI-CCA-2024/models/Fine-Tune-Machine-Unlearning-R1-200.json"
path_modelweights = "/home/yupaporn/codes/Machine-Unlearning-USAI-CCA-2024/models/Fine-Tune-Machine-Unlearning-R1-200.weights.h5"

In [10]:
##load json and create model --------------------
json_file = open(path_modelJson, 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)
#load weights into new model  -------------------

model.load_weights(path_modelweights)
print("Loaded model from disk")

model.summary()

Loaded model from disk
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 224, 224, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 0
Non-trainable params: 28,544,255
_________________________________________________________________


In [6]:
path_modelJson = "/home/yupaporn/codes/Machine-Unlearning-USAI-CCA-2024/models/Fine-Tune-Machine-Unlearning-V1-R1-200-newdata.json"
path_modelweights = "/home/yupaporn/codes/Machine-Unlearning-USAI-CCA-2024/models/Fine-Tune-Machine-Unlearning-V1-R1-200-newdata.weights.h5"

In [7]:
##load json and create model --------------------
json_file = open(path_modelJson, 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)
#load weights into new model  -------------------

model.load_weights(path_modelweights)
print("Loaded model from disk")

model.summary()

Loaded model from disk
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 1,663,199
Non-trainable params: 26,881,056
_________________________________________________________________


In [8]:
layers = model.layers[1].layers
layers

 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb42267a780>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb422688f28>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb4226321d0>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb422646630>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb4225d8a90>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb4225efef0>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb4225953c8>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb4225ab828>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb4225bfc88>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb422556128>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb42257a4a8>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb422511860>,
 <efficientnet.model.get_dropout.<locals>.FixedDropout at 0x7fb422522c18>,
 <efficientnet.model.get_

In [72]:
model.input_shape

(None, 456, 456, 3)

In [73]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: global_average_pooling2d | Trainable: False
Layer: dropout | Trainable: False
Layer: prediction_layer | Trainable: False


In [74]:
layers = model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: True
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Layer

In [54]:
input_shape = (model.input_shape[1], model.input_shape[2], model.input_shape[3])   ### Change input shape  
print(input_shape)

### Unfreeze Block4 
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.layers[1].trainable_weights))
layers = model.layers[1].layers
for innerlayer in layers:
    if innerlayer.name.startswith("block4"):
#             print(innerlayer.name)
        innerlayer.trainable = True
## Unfreeze FC layer
fc_layer = model.get_layer("prediction_layer")
fc_layer.trainable = True

print('This is the number of trainable layers '
      'after freezing the block5a_se_excite Layer:', len(model.layers[1].trainable_weights))

(456, 456, 3)
This is the number of trainable layers before freezing the conv base: 0
This is the number of trainable layers after freezing the block5a_se_excite Layer: 91


In [55]:
model.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 1,663,199
Non-trainable params: 26,881,056
_________________________________________________________________


In [56]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: global_average_pooling2d | Trainable: False
Layer: dropout | Trainable: False
Layer: prediction_layer | Trainable: True


In [57]:
layers = model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: True
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Layer

### TEST CODE: Load ML Unlearn unfreezeB4 Model to Finetune Stage: Unfreeze BlockB4-B7

In [5]:
from tensorflow.keras.models import Model, model_from_json

In [7]:
path_modelJson = "/media/tohn/HDD/Model_unlearn/Models_USAI/ML-unlearn/R1/unfreezeB4-B7/models/modelEffNetB5_ML-unlearn_unfreezeB4-B7-R1.json"
path_modelweights = "/media/tohn/HDD/Model_unlearn/Models_USAI/ML-unlearn/R1/unfreezeB4-B7/models/modelEffNetB5_ML-unlearn_unfreezeB4-B7-R1.weights.h5"

##load json and create model --------------------
json_file = open(path_modelJson, 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)
#load weights into new model  -------------------

model.load_weights(path_modelweights)
print("Loaded model from disk")

model.summary()

Loaded model from disk
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 30,735
Non-trainable params: 28,513,520
_________________________________________________________________


In [8]:
model.input_shape

(None, 456, 456, 3)

In [17]:
input_shape = (model.input_shape[1], model.input_shape[2], model.input_shape[3])
input_shape

(456, 456, 3)

In [9]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: global_average_pooling2d | Trainable: True
Layer: dropout | Trainable: True
Layer: prediction_layer | Trainable: True


In [10]:
layers = model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: True
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Layer

### Unfreeze Block4-B7 

In [13]:
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.layers[1].trainable_weights))
layers = model.layers[1].layers
for innerlayer in layers:
    if any(innerlayer.name.startswith(block) for block in ["block4", "block5", "block6", "block7"]):
        innerlayer.trainable = True
    else:
        innerlayer.trainable = False  # Optional: freeze other layers
## Unfreeze FC layer
fc_layer = model.get_layer("prediction_layer")
fc_layer.trainable = True

print('This is the number of trainable layers '
      'after freezing the block5a_se_excite Layer:', len(model.layers[1].trainable_weights))

This is the number of trainable layers before freezing the conv base: 0
This is the number of trainable layers after freezing the block5a_se_excite Layer: 338


In [14]:
model.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 26,859,743
Non-trainable params: 1,684,512
_________________________________________________________________


In [15]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: global_average_pooling2d | Trainable: True
Layer: dropout | Trainable: True
Layer: prediction_layer | Trainable: True


In [16]:
layers = model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: False
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Laye

### TEST CODE: Load ML Unlearn unfreezeB1-B4 Model to Finetune Stage: Unfreeze BlockB1-B4

In [3]:
from tensorflow.keras.models import Model, model_from_json

In [4]:
path_modelJson = "/media/tohn/HDD/Model_unlearn/Models_USAI/ML-unlearn/R1/unfreezeB1-B4/models/modelEffNetB5_ML-unlearn_unfreezeB1-B4-R1.json"
path_modelweights = "/media/tohn/HDD/Model_unlearn/Models_USAI/ML-unlearn/R1/unfreezeB1-B4/models/modelEffNetB5_ML-unlearn_unfreezeB1-B4-R1.weights.h5"

##load json and create model --------------------
json_file = open(path_modelJson, 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)
#load weights into new model  -------------------

model.load_weights(path_modelweights)
print("Loaded model from disk")

model.summary()

Loaded model from disk
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 30,735
Non-trainable params: 28,513,520
_________________________________________________________________


In [5]:
model.input_shape
input_shape = (model.input_shape[1], model.input_shape[2], model.input_shape[3])
input_shape

(456, 456, 3)

In [6]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: global_average_pooling2d | Trainable: True
Layer: dropout | Trainable: True
Layer: prediction_layer | Trainable: True


In [7]:
layers = model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: True
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Layer

In [8]:
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.trainable_weights))

This is the number of trainable layers before freezing the conv base: 2


### Unfreeze Block1-B4

In [9]:
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.layers[1].trainable_weights))
layers = model.layers[1].layers
for innerlayer in layers:
    if any(innerlayer.name.startswith(block) for block in ["block1", "block2", "block3", "block4"]):
        innerlayer.trainable = True
    else:
        innerlayer.trainable = False  # Optional: freeze other layers
## Unfreeze FC layer
fc_layer = model.get_layer("prediction_layer")
fc_layer.trainable = True

print('This is the number of trainable layers '
      'after freezing the block5a_se_excite Layer:', len(model.layers[1].trainable_weights))

This is the number of trainable layers before freezing the conv base: 0
This is the number of trainable layers after freezing the block5a_se_excite Layer: 251


In [10]:
print('This is the number of trainable layers '
      'after freezing the block5a_se_excite Layer:', len(model.trainable_weights))

This is the number of trainable layers after freezing the block5a_se_excite Layer: 253


In [11]:
model.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 2,120,911
Non-trainable params: 26,423,344
_________________________________________________________________


In [12]:
### Check 
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: global_average_pooling2d | Trainable: True
Layer: dropout | Trainable: True
Layer: prediction_layer | Trainable: True


In [13]:
layers = model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: False
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: True
Layer: block1a_bn, Trainable: True
Layer: block1a_activation, Trainable: True
Layer: block1a_se_squeeze, Trainable: True
Layer: block1a_se_reshape, Trainable: True
Layer: block1a_se_reduce, Trainable: True
Layer: block1a_se_expand, Trainable: True
Layer: block1a_se_excite, Trainable: True
Layer: block1a_project_conv, Trainable: True
Layer: block1a_project_bn, Trainable: True
Layer: block1b_dwconv, Trainable: True
Layer: block1b_bn, Trainable: True
Layer: block1b_activation, Trainable: True
Layer: block1b_se_squeeze, Trainable: True
Layer: block1b_se_reshape, Trainable: True
Layer: block1b_se_reduce, Trainable: True
Layer: block1b_se_expand, Trainable: True
Layer: block1b_se_excite, Trainable: True
Layer: block1b_project_conv, Trainable: True
Layer: block1b_project_bn, Trainable: True
Layer: block1b_drop, Tra

### TEST CODE: Load ML Unlearn unfreezeBlock5a_se_excite-Block7 Model to Finetune Stage: Block5a_se_excite-Block7

In [3]:
from tensorflow.keras.models import Model, model_from_json

path_modelJson = "/media/tohn/HDD/Model_unlearn/Models_USAI/ML-unlearn/R1/Block5a_se_excite/models/modelEffNetB5_ML-unlearn_Block5a_se_excite-R1.json"
path_modelweights = "/media/tohn/HDD/Model_unlearn/Models_USAI/ML-unlearn/R1/Block5a_se_excite/models/modelEffNetB5_ML-unlearn_Block5a_se_excite-R1.weights.h5"

##load json and create model --------------------
json_file = open(path_modelJson, 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)
#load weights into new model  -------------------

model.load_weights(path_modelweights)
print("Loaded model from disk")

model.summary()

Loaded model from disk
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 30,735
Non-trainable params: 28,513,520
_________________________________________________________________


In [4]:
model.input_shape
input_shape = (model.input_shape[1], model.input_shape[2], model.input_shape[3])
input_shape

(456, 456, 3)

In [5]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: global_average_pooling2d | Trainable: True
Layer: dropout | Trainable: True
Layer: prediction_layer | Trainable: True


In [6]:
layers = model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: True
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Layer

### Unfreeze Block5-B7

In [7]:
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.layers[1].trainable_weights))

layers = model.layers[1].layers
## Set efficientnet-b5 to Unfreeze
model.layers[1].trainable = True
set_trainable = False
for innerlayer in layers:
    if innerlayer.name == 'block5a_se_excite':
        set_trainable = True
    if set_trainable:
        innerlayer.trainable = True
    else:
        innerlayer.trainable = False
## Unfreeze FC layer
fc_layer = model.get_layer("prediction_layer")
fc_layer.trainable = True

print('This is the number of trainable layers '
      'after freezing the block5a_se_excite Layer:', len(model.layers[1].trainable_weights))

This is the number of trainable layers before freezing the conv base: 0
This is the number of trainable layers after freezing the block5a_se_excite Layer: 240


In [9]:
print('This is the number of trainable layers '
      'after freezing the block5a_se_excite Layer:', len(model.trainable_weights))

This is the number of trainable layers after freezing the block5a_se_excite Layer: 242


In [10]:
model.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 26,109,423
Non-trainable params: 2,434,832
_________________________________________________________________


In [11]:
### Check 
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: True
Layer: global_average_pooling2d | Trainable: True
Layer: dropout | Trainable: True
Layer: prediction_layer | Trainable: True


In [12]:
layers = model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: False
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Laye

----------------

In [66]:
new_model.summary()

Model: "model_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d_3 ( (None, 2048)              0         
_________________________________________________________________
dropout_3 (Dropout)          (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 30,735
Non-trainable params: 28,513,520
_________________________________________________________________


In [67]:
for layer in new_model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: False
Layer: global_average_pooling2d_3 | Trainable: True
Layer: dropout_3 | Trainable: True
Layer: prediction_layer | Trainable: True


In [68]:
layers = new_model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: False
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Laye

In [61]:
### Unfreeze Block4 
print('This is the number of trainable layers '
      'before freezing the conv base:', len(new_model.layers[1].trainable_weights))
layers = new_model.layers[1].layers
for innerlayer in layers:
    if innerlayer.name.startswith("block4"):
#             print(innerlayer.name)
        innerlayer.trainable = True
print('This is the number of trainable layers '
      'after freezing the Block4 Layer:', len(new_model.layers[1].trainable_weights))

This is the number of trainable layers before freezing the conv base: 0
This is the number of trainable layers after freezing the Block4 Layer: 0


In [62]:
for layer in new_model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: new_input | Trainable: True
Layer: efficientnet-b5 | Trainable: False
Layer: global_average_pooling2d_3 | Trainable: True
Layer: dropout_3 | Trainable: True
Layer: prediction_layer | Trainable: True


In [63]:
layers = new_model.layers[1].layers
for innerlayer in layers:
    print(f"Layer: {innerlayer.name}, Trainable: {innerlayer.trainable}")

Layer: input_4, Trainable: False
Layer: stem_conv, Trainable: False
Layer: stem_bn, Trainable: False
Layer: stem_activation, Trainable: False
Layer: block1a_dwconv, Trainable: False
Layer: block1a_bn, Trainable: False
Layer: block1a_activation, Trainable: False
Layer: block1a_se_squeeze, Trainable: False
Layer: block1a_se_reshape, Trainable: False
Layer: block1a_se_reduce, Trainable: False
Layer: block1a_se_expand, Trainable: False
Layer: block1a_se_excite, Trainable: False
Layer: block1a_project_conv, Trainable: False
Layer: block1a_project_bn, Trainable: False
Layer: block1b_dwconv, Trainable: False
Layer: block1b_bn, Trainable: False
Layer: block1b_activation, Trainable: False
Layer: block1b_se_squeeze, Trainable: False
Layer: block1b_se_reshape, Trainable: False
Layer: block1b_se_reduce, Trainable: False
Layer: block1b_se_expand, Trainable: False
Layer: block1b_se_excite, Trainable: False
Layer: block1b_project_conv, Trainable: False
Layer: block1b_project_bn, Trainable: False
Laye

In [64]:
new_model.summary()

Model: "model_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
new_input (InputLayer)       [(None, 456, 456, 3)]     0         
_________________________________________________________________
efficientnet-b5 (Functional) (None, None, None, 2048)  28513520  
_________________________________________________________________
global_average_pooling2d_3 ( (None, 2048)              0         
_________________________________________________________________
dropout_3 (Dropout)          (None, 2048)              0         
_________________________________________________________________
prediction_layer (Dense)     (None, 15)                30735     
Total params: 28,544,255
Trainable params: 30,735
Non-trainable params: 28,513,520
_________________________________________________________________


In [65]:
## Unfreeze FC layer
fc_layer = new_model.get_layer("prediction_layer")
fc_layer.trainable = True

---------------------

## Code 2021 

In [4]:
from tensorflow.keras.applications import EfficientNetB5 as Net
from efficientnet.keras import center_crop_and_resize, preprocess_input

In [5]:
conv_base = Net(weights='imagenet')
height = width = conv_base.input_shape[1]
input_shape = (height, width, 3)

# loading pretrained conv base model
conv_base = Net(weights='imagenet', include_top=False, input_shape=input_shape)

# create new model with a new classification layer
x = conv_base.output  
global_average_layer = layers.GlobalAveragePooling2D(name = 'head_pooling')(x)
dropout_layer_1 = layers.Dropout(0.50,name = 'head_dropout')(global_average_layer)
prediction_layer = layers.Dense(15, activation='softmax',name = 'prediction_layer')(dropout_layer_1)

model = models.Model(inputs= conv_base.input, outputs=prediction_layer) 
model.summary()

#showing before&after freezing
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.trainable_weights))
#conv_base.trainable = False  # freeze เพื่อรักษา convolutional base's weight
for layer in conv_base.layers:
    layer.trainable = False
print('This is the number of trainable layers '
      'after freezing the conv base:', len(model.trainable_weights))  #freez แล้วจะเหลือ max pool and dense

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_2 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
rescaling_1 (Rescaling)         (None, 456, 456, 3)  0           input_2[0][0]                    
__________________________________________________________________________________________________
normalization_1 (Normalization) (None, 456, 456, 3)  7           rescaling_1[0][0]                
__________________________________________________________________________________________________
stem_conv_pad (ZeroPadding2D)   (None, 457, 457, 3)  0           normalization_1[0][0]            
______________________________________________________________________________________________

block7b_project_conv (Conv2D)   (None, 15, 15, 512)  1572864     block7b_se_excite[0][0]          
__________________________________________________________________________________________________
block7b_project_bn (BatchNormal (None, 15, 15, 512)  2048        block7b_project_conv[0][0]       
__________________________________________________________________________________________________
block7b_drop (Dropout)          (None, 15, 15, 512)  0           block7b_project_bn[0][0]         
__________________________________________________________________________________________________
block7b_add (Add)               (None, 15, 15, 512)  0           block7b_drop[0][0]               
                                                                 block7a_project_bn[0][0]         
__________________________________________________________________________________________________
block7c_expand_conv (Conv2D)    (None, 15, 15, 3072) 1572864     block7b_add[0][0]                
__________

In [7]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: input_2 | Trainable: False
Layer: rescaling_1 | Trainable: False
Layer: normalization_1 | Trainable: False
Layer: stem_conv_pad | Trainable: False
Layer: stem_conv | Trainable: False
Layer: stem_bn | Trainable: False
Layer: stem_activation | Trainable: False
Layer: block1a_dwconv | Trainable: False
Layer: block1a_bn | Trainable: False
Layer: block1a_activation | Trainable: False
Layer: block1a_se_squeeze | Trainable: False
Layer: block1a_se_reshape | Trainable: False
Layer: block1a_se_reduce | Trainable: False
Layer: block1a_se_expand | Trainable: False
Layer: block1a_se_excite | Trainable: False
Layer: block1a_project_conv | Trainable: False
Layer: block1a_project_bn | Trainable: False
Layer: block1b_dwconv | Trainable: False
Layer: block1b_bn | Trainable: False
Layer: block1b_activation | Trainable: False
Layer: block1b_se_squeeze | Trainable: False
Layer: block1b_se_reshape | Trainable: False
Layer: block1b_se_reduce | Trainable: False
Layer: block1b_se_expand | Trainable: Fa

------------

In [3]:
import efficientnet.tfkeras as efn

In [11]:
conv_base = efn.EfficientNetB5(weights='imagenet')
height = width = conv_base.input_shape[1]
input_shape = (height, width, 3)

# loading pretrained conv base model
conv_base = efn.EfficientNetB5(weights='imagenet', include_top=False, input_shape=input_shape)

# create new model with a new classification layer
x = conv_base.output  
global_average_layer = layers.GlobalAveragePooling2D(name = 'head_pooling')(x)
dropout_layer_1 = layers.Dropout(0.50,name = 'head_dropout')(global_average_layer)
prediction_layer = layers.Dense(15, activation='softmax',name = 'prediction_layer')(dropout_layer_1)

model = models.Model(inputs= conv_base.input, outputs=prediction_layer) 
model.summary()

#showing before&after freezing
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.trainable_weights))
#conv_base.trainable = False  # freeze เพื่อรักษา convolutional base's weight
for layer in conv_base.layers:
    layer.trainable = False
print('This is the number of trainable layers '
      'after freezing the conv base:', len(model.trainable_weights))  #freez แล้วจะเหลือ max pool and dense

Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_5 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
stem_conv (Conv2D)              (None, 228, 228, 48) 1296        input_5[0][0]                    
__________________________________________________________________________________________________
stem_bn (BatchNormalization)    (None, 228, 228, 48) 192         stem_conv[0][0]                  
__________________________________________________________________________________________________
stem_activation (Activation)    (None, 228, 228, 48) 0           stem_bn[0][0]                    
____________________________________________________________________________________________

Total params: 28,544,255
Trainable params: 28,371,519
Non-trainable params: 172,736
__________________________________________________________________________________________________
This is the number of trainable layers before freezing the conv base: 506
This is the number of trainable layers after freezing the conv base: 2


In [4]:
# conv_base = efn.EfficientNetB5(weights='imagenet')
# height = width = conv_base.input_shape[1]
# input_shape = (height, width, 3)
input_shape = (456, 456, 3)
# loading pretrained conv base model
conv_base = efn.EfficientNetB5(weights='imagenet', include_top=False, input_shape=input_shape)

# create new model with a new classification layer
x = conv_base.output  
global_average_layer = layers.GlobalAveragePooling2D(name = 'head_pooling')(x)
dropout_layer_1 = layers.Dropout(0.50,name = 'head_dropout')(global_average_layer)
prediction_layer = layers.Dense(15, activation='softmax',name = 'prediction_layer')(dropout_layer_1)

model = models.Model(inputs= conv_base.input, outputs=prediction_layer) 
model.summary()

#showing before&after freezing
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.trainable_weights))
#conv_base.trainable = False  # freeze เพื่อรักษา convolutional base's weight
for layer in conv_base.layers:
    layer.trainable = False
print('This is the number of trainable layers '
      'after freezing the conv base:', len(model.trainable_weights))  #freez แล้วจะเหลือ max pool and dense

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
stem_conv (Conv2D)              (None, 228, 228, 48) 1296        input_1[0][0]                    
__________________________________________________________________________________________________
stem_bn (BatchNormalization)    (None, 228, 228, 48) 192         stem_conv[0][0]                  
__________________________________________________________________________________________________
stem_activation (Activation)    (None, 228, 228, 48) 0           stem_bn[0][0]                    
______________________________________________________________________________________________

Total params: 28,544,255
Trainable params: 28,371,519
Non-trainable params: 172,736
__________________________________________________________________________________________________
This is the number of trainable layers before freezing the conv base: 506
This is the number of trainable layers after freezing the conv base: 2


In [5]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: input_1 | Trainable: False
Layer: stem_conv | Trainable: False
Layer: stem_bn | Trainable: False
Layer: stem_activation | Trainable: False
Layer: block1a_dwconv | Trainable: False
Layer: block1a_bn | Trainable: False
Layer: block1a_activation | Trainable: False
Layer: block1a_se_squeeze | Trainable: False
Layer: block1a_se_reshape | Trainable: False
Layer: block1a_se_reduce | Trainable: False
Layer: block1a_se_expand | Trainable: False
Layer: block1a_se_excite | Trainable: False
Layer: block1a_project_conv | Trainable: False
Layer: block1a_project_bn | Trainable: False
Layer: block1b_dwconv | Trainable: False
Layer: block1b_bn | Trainable: False
Layer: block1b_activation | Trainable: False
Layer: block1b_se_squeeze | Trainable: False
Layer: block1b_se_reshape | Trainable: False
Layer: block1b_se_reduce | Trainable: False
Layer: block1b_se_expand | Trainable: False
Layer: block1b_se_excite | Trainable: False
Layer: block1b_project_conv | Trainable: False
Layer: block1b_project_bn

In [6]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
stem_conv (Conv2D)              (None, 228, 228, 48) 1296        input_1[0][0]                    
__________________________________________________________________________________________________
stem_bn (BatchNormalization)    (None, 228, 228, 48) 192         stem_conv[0][0]                  
__________________________________________________________________________________________________
stem_activation (Activation)    (None, 228, 228, 48) 0           stem_bn[0][0]                    
______________________________________________________________________________________________

-----------------

In [4]:
from tensorflow.keras.applications import EfficientNetB5 as Net
from efficientnet.keras import center_crop_and_resize, preprocess_input
height=width = 456
input_shape = (height, width, 3)
# loading pretrained conv base model
conv_base = Net(weights='imagenet', include_top=False, input_shape=input_shape)
# create new model with a new classification layer
x = conv_base.output  
global_average_layer = layers.GlobalAveragePooling2D(name = 'head_pooling')(x)
# Adding BatchNormalization after pooling
batch_norm_layer = layers.BatchNormalization(name='head_batchnorm')(global_average_layer)
# Dropout layer after BatchNormalization
dropout_layer_1 = layers.Dropout(0.50,name = 'head_dropout')(batch_norm_layer)
# Final classification layer
prediction_layer = layers.Dense(15, activation='softmax',name = 'prediction_layer')(dropout_layer_1)
# Combine layers to define the model
model = models.Model(inputs= conv_base.input, outputs=prediction_layer) 
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_2 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
rescaling_1 (Rescaling)         (None, 456, 456, 3)  0           input_2[0][0]                    
__________________________________________________________________________________________________
normalization_1 (Normalization) (None, 456, 456, 3)  7           rescaling_1[0][0]                
__________________________________________________________________________________________________
stem_conv_pad (ZeroPadding2D)   (None, 457, 457, 3)  0           normalization_1[0][0]            
______________________________________________________________________________________________

In [5]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: input_2 | Trainable: True
Layer: rescaling_1 | Trainable: True
Layer: normalization_1 | Trainable: True
Layer: stem_conv_pad | Trainable: True
Layer: stem_conv | Trainable: True
Layer: stem_bn | Trainable: True
Layer: stem_activation | Trainable: True
Layer: block1a_dwconv | Trainable: True
Layer: block1a_bn | Trainable: True
Layer: block1a_activation | Trainable: True
Layer: block1a_se_squeeze | Trainable: True
Layer: block1a_se_reshape | Trainable: True
Layer: block1a_se_reduce | Trainable: True
Layer: block1a_se_expand | Trainable: True
Layer: block1a_se_excite | Trainable: True
Layer: block1a_project_conv | Trainable: True
Layer: block1a_project_bn | Trainable: True
Layer: block1b_dwconv | Trainable: True
Layer: block1b_bn | Trainable: True
Layer: block1b_activation | Trainable: True
Layer: block1b_se_squeeze | Trainable: True
Layer: block1b_se_reshape | Trainable: True
Layer: block1b_se_reduce | Trainable: True
Layer: block1b_se_expand | Trainable: True
Layer: block1b_se_ex

In [6]:
print('This is the number of trainable layers '
      'before freezing the conv base:', len(model.trainable_weights))  

This is the number of trainable layers before freezing the conv base: 508


In [7]:
#Unfreez
model.trainable = True
set_trainable = False
for layer in model.layers:
    if layer.name == 'block5a_se_excite':
        set_trainable = True
    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False
print('This is the number of trainable layers '
      'after freezing the conv base:', len(model.trainable_weights))  

This is the number of trainable layers after freezing the conv base: 244


In [8]:
for layer in model.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

Layer: input_2 | Trainable: False
Layer: rescaling_1 | Trainable: False
Layer: normalization_1 | Trainable: False
Layer: stem_conv_pad | Trainable: False
Layer: stem_conv | Trainable: False
Layer: stem_bn | Trainable: False
Layer: stem_activation | Trainable: False
Layer: block1a_dwconv | Trainable: False
Layer: block1a_bn | Trainable: False
Layer: block1a_activation | Trainable: False
Layer: block1a_se_squeeze | Trainable: False
Layer: block1a_se_reshape | Trainable: False
Layer: block1a_se_reduce | Trainable: False
Layer: block1a_se_expand | Trainable: False
Layer: block1a_se_excite | Trainable: False
Layer: block1a_project_conv | Trainable: False
Layer: block1a_project_bn | Trainable: False
Layer: block1b_dwconv | Trainable: False
Layer: block1b_bn | Trainable: False
Layer: block1b_activation | Trainable: False
Layer: block1b_se_squeeze | Trainable: False
Layer: block1b_se_reshape | Trainable: False
Layer: block1b_se_reduce | Trainable: False
Layer: block1b_se_expand | Trainable: Fa

In [9]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_2 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
rescaling_1 (Rescaling)         (None, 456, 456, 3)  0           input_2[0][0]                    
__________________________________________________________________________________________________
normalization_1 (Normalization) (None, 456, 456, 3)  7           rescaling_1[0][0]                
__________________________________________________________________________________________________
stem_conv_pad (ZeroPadding2D)   (None, 457, 457, 3)  0           normalization_1[0][0]            
______________________________________________________________________________________________

------------

In [3]:
model_dir = "/media/tohn/HDD/Model_unlearn/Models_USAI/expV2/R2_balanceclass/on_epoch_end/modelEffNetB5_base_Block5a_se_excite_newdatabalance-R2_last.h5"
model = load_model(model_dir)
print(f"Load model to resume: {model_dir}")
print("="*125)
height = width = model.input_shape[0] 
model.summary()

Load model to resume: /media/tohn/HDD/Model_unlearn/Models_USAI/expV2/R2_balanceclass/on_epoch_end/modelEffNetB5_base_Block5a_se_excite_newdatabalance-R2_last.h5
Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
rescaling (Rescaling)           (None, 456, 456, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
normalization (Normalization)   (None, 456, 456, 3)  7           rescaling[0][0]                  
__________________________________________________________________________________________________
stem_conv_pad (ZeroPadding2D)  

block5a_se_expand (Conv2D)      (None, 1, 1, 768)    25344       block5a_se_reduce[0][0]          
__________________________________________________________________________________________________
block5a_se_excite (Multiply)    (None, 29, 29, 768)  0           block5a_activation[0][0]         
                                                                 block5a_se_expand[0][0]          
__________________________________________________________________________________________________
block5a_project_conv (Conv2D)   (None, 29, 29, 176)  135168      block5a_se_excite[0][0]          
__________________________________________________________________________________________________
block5a_project_bn (BatchNormal (None, 29, 29, 176)  704         block5a_project_conv[0][0]       
__________________________________________________________________________________________________
block5b_expand_conv (Conv2D)    (None, 29, 29, 1056) 185856      block5a_project_bn[0][0]         
__________

In [4]:
print(height, width )

None None


In [5]:
model.input_shape

(None, 456, 456, 3)

In [6]:
height = width = model.input_shape[1] 

In [7]:
print(height, width )

456 456


----------------------------------------------

## efficientnet ==>  weights (random initial weights) 

In [3]:
import efficientnet.tfkeras as efn

In [4]:
imagenet_base = efn.EfficientNetB5(weights='imagenet')
imagenet_base.summary()

Model: "efficientnet-b5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
stem_conv (Conv2D)              (None, 228, 228, 48) 1296        input_1[0][0]                    
__________________________________________________________________________________________________
stem_bn (BatchNormalization)    (None, 228, 228, 48) 192         stem_conv[0][0]                  
__________________________________________________________________________________________________
stem_activation (Activation)    (None, 228, 228, 48) 0           stem_bn[0][0]                    
____________________________________________________________________________________

In [6]:
conv_base = efn.EfficientNetB5(weights=None)
conv_base.summary()

Model: "efficientnet-b5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_2 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
stem_conv (Conv2D)              (None, 228, 228, 48) 1296        input_2[0][0]                    
__________________________________________________________________________________________________
stem_bn (BatchNormalization)    (None, 228, 228, 48) 192         stem_conv[0][0]                  
__________________________________________________________________________________________________
stem_activation (Activation)    (None, 228, 228, 48) 0           stem_bn[0][0]                    
____________________________________________________________________________________

Total params: 30,562,520
Trainable params: 30,389,784
Non-trainable params: 172,736
__________________________________________________________________________________________________


In [12]:
np.array_equal(imagenet_base.weights[0].numpy(), conv_base.weights[0].numpy())

False

In [14]:
imagenet_base.weights[0].numpy()

array([[[[ 1.18231654e-01,  4.25131619e-01, -1.23313434e-01, ...,
           2.23586783e-02, -4.54221547e-01,  5.53142279e-03],
         [ 1.19787961e-01,  9.77019310e-01, -1.68859288e-01, ...,
           3.59148979e-02, -1.01231885e+00, -2.35486180e-01],
         [ 3.81877236e-02,  3.13481152e-01, -6.57095090e-02, ...,
          -1.97226778e-02, -3.82235736e-01,  3.49403732e-02]],

        [[-1.73283100e-01,  3.77537310e-02, -4.12713289e-01, ...,
          -2.15455890e-02, -2.17378020e-01, -1.14280093e+00],
         [-3.09628487e-01,  7.50746503e-02, -3.41504902e-01, ...,
           5.16232848e-02, -1.29302144e-01, -2.83064914e+00],
         [-6.32216483e-02, -1.20263286e-02, -9.80676040e-02, ...,
          -5.96328266e-03, -4.59197089e-02, -2.60618836e-01]],

        [[-2.87676975e-03, -2.02377662e-02, -8.32590908e-02, ...,
           2.83462182e-02,  1.78298876e-02, -8.29558894e-02],
         [ 1.14317164e-01,  4.18053195e-02, -2.38755345e-02, ...,
           3.54998223e-02,  4.4169

In [15]:
conv_base.weights[0].numpy()

array([[[[ 9.23638195e-02,  1.18085556e-01,  5.49152680e-02, ...,
           6.14659265e-02,  6.73452765e-02,  2.91241296e-02],
         [ 1.03188083e-01, -9.60463658e-02,  1.08607054e-01, ...,
           1.63944587e-02, -1.92539040e-02,  3.23935226e-02],
         [ 4.12125289e-02,  3.53991091e-02,  2.34960504e-02, ...,
          -9.89319608e-02, -1.43191573e-04, -3.25513892e-02]],

        [[-6.16268702e-02,  1.91030167e-02,  3.34922560e-02, ...,
          -9.99613553e-02, -1.48444891e-01,  3.80052552e-02],
         [ 5.84926121e-02,  7.44147077e-02, -1.39274716e-01, ...,
          -3.87874171e-02, -2.65908390e-02, -3.32620740e-02],
         [-5.56417648e-03,  4.43095826e-02, -1.19205378e-01, ...,
          -3.58318686e-02, -4.14046720e-02, -5.47261722e-02]],

        [[-8.42836499e-02, -3.99682950e-03, -3.50278839e-02, ...,
           4.59800921e-02, -5.16069196e-02, -7.02959299e-02],
         [-1.58630330e-02,  6.36456208e-03,  5.35517111e-02, ...,
          -8.25815871e-02, -4.2127

In [17]:
print(conv_base.input_shape[1:])

(456, 456, 3)


In [18]:
input_shape = conv_base.input_shape[1:]
input_shape

(456, 456, 3)

In [16]:
print(conv_base.input[1])
print(conv_base.input[2])
print(conv_base.input[3])

KerasTensor(type_spec=TensorSpec(shape=(456, 456, 3), dtype=tf.float32, name=None), name='tf.__operators__.getitem_4/strided_slice:0', description="created by layer 'tf.__operators__.getitem_4'")
KerasTensor(type_spec=TensorSpec(shape=(456, 456, 3), dtype=tf.float32, name=None), name='tf.__operators__.getitem_5/strided_slice:0', description="created by layer 'tf.__operators__.getitem_5'")
KerasTensor(type_spec=TensorSpec(shape=(456, 456, 3), dtype=tf.float32, name=None), name='tf.__operators__.getitem_6/strided_slice:0', description="created by layer 'tf.__operators__.getitem_6'")


In [4]:
weight = None
print(weight)
conv_base_v2 = efn.EfficientNetB5(weights=weight)
conv_base_v2.summary()

None
Model: "efficientnet-b5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
stem_conv (Conv2D)              (None, 228, 228, 48) 1296        input_1[0][0]                    
__________________________________________________________________________________________________
stem_bn (BatchNormalization)    (None, 228, 228, 48) 192         stem_conv[0][0]                  
__________________________________________________________________________________________________
stem_activation (Activation)    (None, 228, 228, 48) 0           stem_bn[0][0]                    
_______________________________________________________________________________

In [20]:
np.array_equal(conv_base_v2.weights[0].numpy(), conv_base.weights[0].numpy())

False

In [22]:
weight = 'imagenet'
print(weight)
imagenet_base_v2 = efn.EfficientNetB5(weights=weight)
imagenet_base_v2.summary()

imagenet
Model: "efficientnet-b5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_5 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
stem_conv (Conv2D)              (None, 228, 228, 48) 1296        input_5[0][0]                    
__________________________________________________________________________________________________
stem_bn (BatchNormalization)    (None, 228, 228, 48) 192         stem_conv[0][0]                  
__________________________________________________________________________________________________
stem_activation (Activation)    (None, 228, 228, 48) 0           stem_bn[0][0]                    
___________________________________________________________________________

Total params: 30,562,520
Trainable params: 30,389,784
Non-trainable params: 172,736
__________________________________________________________________________________________________


> ### Save model

In [14]:
import efficientnet.tfkeras as efn
from CNNs_unlearn.EffNetmodel import model_block5a_se_excite_Unfreze, build_modelB5_unlearn

init_weight = None
input_shape, conv_base_v2 = build_modelB5_unlearn(fine_tune=True, init_weight=init_weight)
#conv_base_v2 = efn.EfficientNetB5(weights=weight)
conv_base_v2.summary()

This is the number of trainable layers before freezing the conv base: 510
[INFO]: Freezing hidden layers...
This is the number of trainable layers after freezing the conv base: 6
-----------------------------------------------------------------------------------------------------------------------------
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 456, 456, 3  0           []                               
                                )]                                                                
                                                                                                  
 input_4 (InputLayer)           [(None, 456, 456, 3  0           []                               
                                )]                                                    

In [15]:
input_shape

(456, 456, 3)

In [16]:
IMAGE_SIZE = input_shape[0]
print(IMAGE_SIZE)

456


In [17]:
'''
Function: Calculate step number for each epoch for training model unlearning.
'''
def cal_steps(num_images, batch_size):
    # calculates steps for generator
    steps = num_images // batch_size
   # adds 1 to the generator steps if the steps multiplied by
   # the batch size is less than the total training samples
    return steps + 1 if (steps * batch_size) < num_images else steps

In [18]:
from CNNs_unlearn.data_generator import batch_datagen, Flip_generator

In [19]:
data_path = "/home/kannika/code/mini-ImageNet_MachineUnlearn.csv"
batchsize = 4
## Create Data Training Loader
dataset = pd.read_csv(data_path, dtype=str)
Train_df = dataset[dataset['subset']=='train'].reset_index(drop=True)        
val_df = dataset[dataset['subset']=='val'].reset_index(drop=True)
### Implement > ## Train set  
batch_generator_main, batch_generator_2 = batch_datagen(Train_df, input_shape, BATCH_SIZE=batchsize)
batch_train = Flip_generator(batch_generator_main, batch_generator_2, IMAGE_SIZE=IMAGE_SIZE)
## Validation set 
batchval_generator_main, batchval_generator_2 = batch_datagen(val_df, input_shape, BATCH_SIZE=batchsize)
batch_val = Flip_generator(batchval_generator_main, batchval_generator_2, IMAGE_SIZE=IMAGE_SIZE)

Found 42000 validated image filenames.
Found 42000 validated image filenames belonging to 2 classes.
Found 12000 validated image filenames.
Found 12000 validated image filenames belonging to 2 classes.


In [20]:
### Set shows train steps:
steps_per_epoch = cal_steps(len(Train_df), batch_size=batchsize)
print(steps_per_epoch)
validation_steps = cal_steps(len(val_df), batch_size=batchsize)
print(validation_steps)

10500
3000


In [21]:
### Set loss and opimizer 
loss_smooth = tf.keras.losses.BinaryCrossentropy(label_smoothing=0.2)
opt = tf.keras.optimizers.legacy.Adam(learning_rate=1e-5, decay=1e-4)
#Compliermodel
conv_base_v2.compile(optimizer=opt, loss=loss_smooth, metrics=['accuracy'])

In [22]:
# Fit model
epochs = 2
conv_base_v2.fit(batch_train, steps_per_epoch=steps_per_epoch, epochs=epochs, batch_size=batchsize,
                    validation_data=batch_val, validation_steps=validation_steps)

Epoch 1/2


2025-05-27 11:45:07.797856: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inmodel/efficientnet-b5/block1b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-05-27 11:45:10.061778: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8100
2025-05-27 11:45:10.444559: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


   31/10500 [..............................] - ETA: 43:29 - loss: 0.6931 - accuracy: 0.9355

KeyboardInterrupt: 

In [29]:
conv_base_v2.save(f'{root_metrics}/{on_epoch_name}_epoch{epochs}', save_format="tf")

INFO:tensorflow:Assets written to: /media/tohn/HDD2/Model_unlearn/EffNetB5Model/baseML_unlearn/weight_random/R1/transfer/on_epoch_end/modelEffNetB5_Unlearning_miniImageNet_transfer-R1_epoch1/assets


INFO:tensorflow:Assets written to: /media/tohn/HDD2/Model_unlearn/EffNetB5Model/baseML_unlearn/weight_random/R1/transfer/on_epoch_end/modelEffNetB5_Unlearning_miniImageNet_transfer-R1_epoch1/assets


> ## Testing: predict

In [3]:
TESTdf = pd.read_csv("/home/kannika/code/DATATEST-V2_mini_ImageNet_MachineUnlearn.csv", dtype=str)
TESTdf01 = TESTdf.iloc[:2000,:]
TESTdf01 = TESTdf01.reset_index(drop=True)
print(TESTdf01.shape)
TESTdf01.tail()

(2000, 8)


,Unnamed: 0.1,Unnamed: 0,filename,img_path_main,img_path_2,cls,encode_label,subset
1995,1995,1995,n0213844100001199,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,FALSE,0,test
1996,1996,1996,n0213844100000746,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,FALSE,0,test
1997,1997,1997,n0213844100000842,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,FALSE,0,test
1998,1998,1998,n0213844100000232,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,FALSE,0,test
1999,1999,1999,n0213844100000277,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,/media/tohn/HDD2/miniImageNet_Dataset/mini-ima...,FALSE,0,test


In [4]:
height = width = 456

In [5]:
from tensorflow.keras.preprocessing import image

def TENSOR_image(img_path):
    # Read the image and resize it
    img = image.load_img(img_path, target_size=(height, width))
    # Convert it to a Numpy array with target shape.
    x = image.img_to_array(img)
    # Reshape
    x = x.reshape((1,) + x.shape)
    x /= 255.

    return x


def preparebatch(TESTdf):
    listimg_test = []
    for j in range(len(TESTdf)):
        pathimgmain = TESTdf["img_path_main"][j]
        x_main = TENSOR_image(pathimgmain)
        pathimg2 = TESTdf["img_path_2"][j]
        x_2 = TENSOR_image(pathimg2)
        img_test = [x_main, x_2]
        listimg_test.append(img_test)
        
    return listimg_test

def predicted(listimg_test):
    pred_list = list()
    prob_list = list()

    for i in range(0, len(listimg_test)):
        predict = model.predict(listimg_test[i]) 
        predict0 = predict[0][0]
        prob_list.append(predict0)
        if predict0 > 0.5:
            pred_list.append("TRUE")
        else:
            pred_list.append("FALSE")
            
    return pred_list, prob_list

In [6]:
listimg_test01=preparebatch(TESTdf01)
print(len(listimg_test01))

2000


In [7]:
from tensorflow import keras

model = keras.models.load_model('/media/tohn/HDD2/Model_unlearn/EffNetB5Model/baseML_unlearn/weight_random/R1/transfer/models/modelEffNetB5_Unlearning_miniImageNet_transfer-R1')
model.summary()

2025-05-27 14:35:11.248797: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-27 14:35:11.649571: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9619 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:17:00.0, compute capability: 7.5


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 456, 456, 3  0           []                               
                                )]                                                                
                                                                                                  
 input_3 (InputLayer)           [(None, 456, 456, 3  0           []                               
                                )]                                                                
                                                                                                  
 concatenate (Concatenate)      (None, 456, 456, 6)  0           ['input_2[0][0]',                
                                                                  'input_3[0][0]']            

In [8]:
pred_list01, prob_list01 = predicted(listimg_test01)

2025-05-27 14:35:36.346291: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8100
2025-05-27 14:35:36.697702: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


1/1 [==============================] - 0s 45ms/step


1/1 [==============================] - 0s 46ms/step


1/1 [==============================] - 0s 38ms/step


1/1 [==============================] - 0s 51ms/step


...

1/1 [==============================] - 0s 56ms/step


1/1 [==============================] - 0s 39ms/step


1/1 [==============================] - 0s 47ms/step


1/1 [==============================] - 0s 38ms/step


1/1 [==============================] - 0s 44ms/step


In [9]:
print(len(pred_list01))
print(len(prob_list01))

2000
2000


In [10]:
pred_list01

['TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 'TRUE',
 

In [48]:
prob_list01

[0.5000783,
 0.5000792,
 0.5000768,
 0.5000786,
 0.5000791,
 0.50007784,
 0.50007874,
 0.5000782,
 0.50007576,
 0.50007796,
 0.5000801,
 0.500078,
 0.5000783,
 0.5000782,
 0.5000779,
 0.50007766,
 0.5000781,
 0.50007814,
 0.5000785,
 0.50007915,
 0.50007653,
 0.5000785,
 0.50007766,
 0.5000789,
 0.5000781,
 0.50007844,
 0.50007915,
 0.5000776,
 0.5000789,
 0.5000787,
 0.50007737,
 0.5000782,
 0.50007814,
 0.50007784,
 0.5000788,
 0.5000781,
 0.50007874,
 0.50007683,
 0.50007904,
 0.5000782,
 0.5000784,
 0.5000794,
 0.50007856,
 0.5000785,
 0.50008,
 0.500078,
 0.50008005,
 0.5000779,
 0.5000793,
 0.500079,
 0.50007826,
 0.5000788,
 0.5000787,
 0.50007904,
 0.50007874,
 0.5000783,
 0.5000793,
 0.5000786,
 0.50007784,
 0.5000801,
 0.50007814,
 0.5000798,
 0.500079,
 0.5000775,
 0.50007874,
 0.5000795,
 0.5000796,
 0.5000789,
 0.5000788,
 0.500079,
 0.5000794,
 0.50007874,
 0.5000783,
 0.5000757,
 0.5000802,
 0.5000795,
 0.50007874,
 0.5000795,
 0.50007856,
 0.5000791,
 0.5000787,
 0.5000

------------------------------------------------------

In [23]:
root_metrics = "/media/tohn/HDD2/Model_unlearn/EffNetB5Model/baseML_unlearn/weight_random/R1/transfer/on_epoch_end"
network_name = 'EffNetB5'
name = 'transfer'
_R = 'R1'
epochs = 1
on_epoch_name = f'model{network_name}_Unlearning_miniImageNet_{name}-{_R}'
on_epoch_name

'modelEffNetB5_Unlearning_miniImageNet_transfer-R1'

In [28]:
conv_base_v2.save(f'{root_metrics}/{on_epoch_name}_epoch{epochs}.keras', save_format="keras")

ValueError: Insufficient precision in available types to represent (31, 23, 8, 0, 23)

In [26]:
f'{root_metrics}/{on_epoch_name}_epoch{epochs}.weights.h5'

'/media/tohn/HDD2/Model_unlearn/EffNetB5Model/baseML_unlearn/weight_random/R1/transfer/on_epoch_end/modelEffNetB5_Unlearning_miniImageNet_transfer-R1_epoch1.weights.h5'

In [27]:
conv_base_v2.save_weights(f'{root_metrics}/{on_epoch_name}_epoch{epochs}.weights.h5')

ValueError: Insufficient precision in available types to represent (31, 23, 8, 0, 23)

-----------------------------

In [4]:
from tensorflow import keras

rootpath = '/media/tohn/HDD2/Model_unlearn/EffNetB5Model/baseML_unlearn/weight_random/R1/transfer'
model = keras.models.load_model(f'{rootpath}/models/modelEffNetB5_Unlearning_miniImageNet_transfer-R1.h5')
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_2 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
input_3 (InputLayer)            [(None, 456, 456, 3) 0                                            
__________________________________________________________________________________________________
concatenate (Concatenate)       (None, 456, 456, 6)  0           input_2[0][0]                    
                                                                 input_3[0][0]                    
__________________________________________________________________________________________________
conv2d (Conv2D)                 (None, 456, 456, 3)  21          concatenate[0][0]            

In [7]:
from CNNs_unlearn.utils import colorstr

print(colorstr('green', f'Save USAI Model as: {rootpath}/models/modelEffNetB5_Unlearning_miniImageNet_transfer-R1.h5'))

Save USAI Model as: /media/tohn/HDD2/Model_unlearn/EffNetB5Model/baseML_unlearn/weight_random/R1/transfer/models/modelEffNetB5_Unlearning_miniImageNet_transfer-R1.h5
